
# NB_OTT_TicketClustering

Ejecuta el clustering operacional sobre los embeddings previamente generados (almacenados en Silver.TicketEmbeddings) y persiste los resultados en Gold.

Los parámetros eps y min_samples se reciben desde Fabric Pipeline. El notebook asigna un cluster_id a cada ticket y persiste los resultados en Gold.TicketClusters y Gold.ClusteredTickets.

No realiza grid search ni utiliza incident_id, ya que la evaluación experimental se mantiene separada del pipeline operacional.


In [1]:
# Parámetros por defecto
eps = 0.25
min_samples = 3
embedding_model = "all-MiniLM-L6-v2"

StatementMeta(, cd78b9e4-68a3-40ca-8da5-52c1aa91acf0, 5, Finished, Available, Finished, False)

In [2]:
# ============================================================
# 1. Preparación de parámetros
# ============================================================

FINAL_EPS = float(eps)
FINAL_MIN_SAMPLES = int(min_samples)

print("Clustering configuration")
print("------------------------")
print("Algorithm: DBSCAN")
print("eps:", FINAL_EPS)
print("min_samples:", FINAL_MIN_SAMPLES)
print("embedding_model:", embedding_model)

StatementMeta(, cd78b9e4-68a3-40ca-8da5-52c1aa91acf0, 6, Finished, Available, Finished, False)

Configuration:
eps: 0.25
min_samples: 3
embedding_model: all-MiniLM-L6-v2


In [ ]:
# ============================================================
# 2. Imports
# ============================================================

import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN
from pyspark.sql import functions as F

In [ ]:
# ============================================================
# 3. Carga de embeddings
#
# Solo son necesarios:
# - ticket_id
# - embedding
#
# El ground truth (incident_id) NO se carga porque no forma parte del proceso operacional sino de los experimentos.
# ============================================================

df_embeddings = (
    spark.table("Silver.TicketEmbeddings")
    .select(
        "ticket_id",
        "embedding"
    )
)

embedding_count = df_embeddings.count()

print("Embeddings loaded:", embedding_count)

display(
    df_embeddings.limit(5)
)

In [ ]:
# ============================================================
# 4. Conversión a matriz NumPy
#
# DBSCAN de scikit-learn trabaja localmente con una matriz NumPy. Cada fila representa un ticket y cada columna una dimensión del embedding.
# ============================================================

pdf_embeddings = df_embeddings.toPandas()

X = np.vstack(
    pdf_embeddings["embedding"]
    .apply(np.array)
    .values
)

print("Embedding matrix shape:", X.shape)

In [ ]:
# ============================================================
# 5. Ejecución del clustering final
#
# Se ejecuta UNA única configuración de DBSCAN.
#
# Los hiperparámetros han sido seleccionados previamente mediante NB_OTT_ExperimentsCluster.
# ============================================================

dbscan = DBSCAN(
    eps=FINAL_EPS,
    min_samples=FINAL_MIN_SAMPLES,
    metric="cosine"
)

cluster_labels = dbscan.fit_predict(X)

pdf_embeddings["cluster_id"] = cluster_labels

In [ ]:
# ============================================================
# 6. Validación operacional básica
#
# Estas métricas NO necesitan ground truth y sirven para comprobar que el algoritmo se ha ejecutado correctamente.
# ============================================================

n_clusters = len(
    set(cluster_labels) - {-1}
)

n_noise = int(
    np.sum(cluster_labels == -1)
)

print("Clustering completed")
print("--------------------")
print("Tickets:", len(pdf_embeddings))
print("Clusters detected:", n_clusters)
print("Noise tickets:", n_noise)

print("\nCluster distribution:")

print(
    pdf_embeddings["cluster_id"]
    .value_counts()
    .sort_values(ascending=False)
)

In [ ]:
# ============================================================
# 7. Persistencia de Gold.TicketClusters
#
# Se almacena únicamente:
# - ticket_id
# - cluster_id
# - configuración utilizada
#
# Los embeddings no se duplican en Gold.
# ============================================================

clusters_pdf = pdf_embeddings[
    [
        "ticket_id",
        "cluster_id"
    ]
].copy()

clusters_pdf["clustering_algorithm"] = "DBSCAN"
clusters_pdf["eps"] = FINAL_EPS
clusters_pdf["min_samples"] = FINAL_MIN_SAMPLES
clusters_pdf["embedding_model"] = embedding_model

df_clusters = (
    spark.createDataFrame(clusters_pdf)
    .withColumn(
        "_processed_at",
        F.current_timestamp()
    )
)

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS Gold"
)

(
    df_clusters.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "Gold.TicketClusters"
    )
)

print(
    "Gold.TicketClusters written:",
    spark.table("Gold.TicketClusters").count()
)

In [ ]:
# ============================================================
# 8. Creación de Gold.ClusteredTickets
#
# Se enriquecen los tickets Silver con su cluster asignado. Esta tabla será utilizada posteriormente para construir IncidentGroups.
# ============================================================

df_tickets = (
    spark.table("Silver.Tickets")
)

df_clusters = (
    spark.table("Gold.TicketClusters")
    .withColumnRenamed(
        "_processed_at",
        "_clustered_at"
    )
)

df_gold = (
    df_tickets
    .join(
        df_clusters,
        on="ticket_id",
        how="inner"
    )
)

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "Gold.ClusteredTickets"
    )
)

print(
    "Gold.ClusteredTickets written:",
    spark.table("Gold.ClusteredTickets").count()
)

In [ ]:
# ============================================================
# 9. Validación final
# ============================================================

display(
    spark.sql("""
        SELECT
            cluster_id,
            COUNT(*) AS ticket_count
        FROM Gold.ClusteredTickets
        GROUP BY cluster_id
        ORDER BY ticket_count DESC
    """)
)

print("NB_OTT_TicketClustering completed successfully.")